**[🏠 Course Home](00_START_HERE.ipynb)** | **📖 [Conceptual](../conceptual/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🤖 [AI-Native Track](README.md)** | **🐍 [Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🔵 [R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)** | **⬅️ [Prev: Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** | **➡️ [Next: Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 20 of 22*

---

# Chapter 20: CI/CD Test Promotion — Streak Heuristics ($N, M$) vs. Bayesian Filtering
### *The Gambler's Fallacy in CI, Survivorship Bias, and Dynamic Memory Promotion Gates*

> **🧭 Course Context & Why This Module Matters in the AI Era:**  
> Autonomous DevOps and CI orchestration agents are increasingly tasked with
> managing test quarantine tiers, promoting new tests to blocking gates, and
> quarantining flaky regressions. When prompted, LLMs almost universally
> suggest naive streak rules: *"Promote after 10 passes, demote after 2
> failures."* This module exposes why streak heuristics suffer from structural
> amnesia and survivorship bias, and teaches autonomous agents how to implement
> dynamic Bayesian discount filters with exponential memory.



## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt an Engineer Types
A DevOps engineer asks an AI assistant to build an automated test promotion
state machine:

```text
"We have a quarantine staging tier for new and patched tests.
Can you write a Python state machine to automatically promote tests from
quarantine to blocking CI, and demote them if they flake?"
```

### The Plausible-Sounding AI Hallucination
The AI generates an intuitive-sounding streak counter rule:

> *"Here is a robust state machine for CI/CD test promotion!
> - **Promotion Rule**: If a test in quarantine achieves **10 consecutive
>   passes**, promote it to the blocking quality gate.
> - **Demotion Rule**: If a blocking test suffers **2 consecutive failures**,
>   demote it back to quarantine.
> This simple (10, 2) streak heuristic prevents premature promotions while
> tolerating occasional transient network glitches!"*

```text
               THE NAIVE (10, 2) STREAK TRAP (AI HALLUCINATION)
               
   Quarantine ──► [Streak == 10 Passes] ──► Promoted to Blocking CI Gate
        ▲                                              │
        │                                              ▼
   Demoted after 2 Consecutive Fails ◄── Broken 3% Flaky Test Remains Promoted
                                         for 300+ Builds (Blocks Developers!)
```

### Forensic Breakdown: Why the AI Failed
The AI's streak state machine suffers from four fatal probabilistic flaws:

1. **The Geometric Trap & Survivorship Bias**:
   Consider a permanently broken test with a **$3\%$ true failure rate** ($p =
   0.03$). The probability that this test achieves 10 consecutive passes on any
   given attempt is:
   $$P(\text{Streak of 10}) = (1 - 0.03)^{10} = (0.97)^{10} \approx \mathbf{73.7\%}$$
   Almost three out of four times, a broken test immediately clears the
   promotion threshold!
   Even under a stricter $(50, 10)$ rule where a 50-pass streak is required:
   $$P(\text{Streak of 50} \mid p = 0.05) = (0.95)^{50} \approx 7.69\%$$
   Over 500 builds in staging, the probability of achieving at least one lucky
   50-pass streak is:
   $$P(\ge 1 \text{ streak in 500}) \approx \mathbf{58.4\%}$$
   More than half of permanently broken tests are guaranteed to get promoted!

2. **Structural Amnesia (The Gambler's Fallacy)**:
   Suppose Test A fails 8 out of 10 runs, then strings together 10 lucky
   passes. Test B runs cleanly for 500 consecutive runs. To the AI's streak
   counter, **both tests look identical**. The heuristic has total amnesia for
   any telemetry that occurred 11 runs ago.

3. **Delayed Demotion & Asymmetric Blast Radius**:
   A $3\%$ flaky test rarely fails twice in a row:
   $$P(\text{Demote}) = (0.03)^2 = 0.0009 \implies 0.09\%$$
   The test will fail isolated builds, disrupt dozens of engineers, and remain
   promoted for hundreds of runs before finally suffering two consecutive
   failures.

4. **The Bayesian Solution (Dynamic Discount State Machine)**:
   A Bayesian system replaces discrete streak counters with a continuous
   $\text{Beta}(\alpha, \beta)$ belief state with exponential memory decay
   $\gamma = 0.98$ ($N_{\text{eff}} = 50$). Tests are promoted if and only if
   $P(\theta \le 1.0\% \mid \text{Telemetry}) \ge 95\%$.



## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. System: CI/CD test promotion gatekeeper tracking individual test flakiness.
2. Flaw: Explain why (N, M) streak counters suffer from Gambler's Fallacy and allow
   5% flaky tests to pass over 500 builds.
3. Bayesian Filter: Implement dynamic Beta-Binomial discount filter (gamma = 0.98,
   N_eff = 50) requiring P(theta <= 1.0%) >= 0.95 and immediate demotion on failure.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Filter:    alpha_t = 0.98 * alpha_{t-1} + k_t,  beta_t = 0.98 * beta_{t-1} + (n_t - k_t)
Promotion: State = PROMOTED iff P(theta <= 0.01 | alpha_t, beta_t) >= 0.95
Demotion:  Demote immediately upon single failure shock
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a CI Gatekeeper Architect. We use a rule where 10 passes promotes a
test and 2 failures demotes it. Ask me 3 questions about survivorship bias, memory
decay, and Bayesian state machines before rewriting our gate."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
specification_contract:
  target: "Dynamic Bayesian Test Promotion and Demotion State Machine"
  module: "Chapter 20 Continuous Quality Gatekeeping"
  
  observables:
    telemetry_stream:
      type: "stream[BuildEvent]"
      fields:
        test_id: "string"
        result: "{PASS, FAIL}"
        timestamp: "ISO8601"
        
  prior_distribution:
    name: "Beta(alpha_0, beta_0)"
    alpha_0: 1.0   # Neutral pseudo-failure count
    beta_0: 99.0   # Prior belief that well-authored tests have ~1% baseline flake
    
  memory_architecture:
    type: "Exponential Memory Decay (Fading Ink)"
    decay_parameter_gamma: 0.98
    effective_sample_size: 50.0  # N_eff = 1 / (1 - gamma)
    half_life_builds: 34.3       # ln(0.5) / ln(gamma)
    
  dynamic_update_rules:
    on_pass:
      alpha_t: "gamma * alpha_{t-1}"
      beta_t: "gamma * beta_{t-1} + 1.0"
    on_fail:
      alpha_t: "gamma * alpha_{t-1} + 1.0"
      beta_t: "gamma * beta_{t-1}"
      
  epistemic_decision_gates:
    target_metric: "Posterior Credible Mass: P(theta <= theta_threshold | alpha_t, beta_t)"
    theta_threshold: 0.010       # 1.0% maximum tolerable flakiness
    promotion_gate:
      condition: "P(theta <= 0.010 | alpha_t, beta_t) >= 0.950"
      action: "PROMOTE from Quarantine to Blocking Gate"
    demotion_gate:
      condition: "P(theta <= 0.010 | alpha_t, beta_t) < 0.800"
      action: "DEMOTE from Blocking Gate to Quarantine immediately"
      
  operational_invariants:
    state_persistence: "O(1) memory overhead; store only (alpha, beta, state)"
    immunity_to_survivorship_bias: "Strictly prevent promotion of theta >= 0.03 tests"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec or Level 4 YAML contract from
Part 2, it rejects integer streak counters and implements a recursive Bayesian
discount filter with exponential memory. Here is the verified mathematical
derivation and execution telemetry:

### Mathematical Derivation of the Dynamic Discount Filter
1. **Prior Distribution**:
   Initial uninformative baseline $\text{Beta}(\alpha_0 = 1, \beta_0 = 1)$ or
   canary initialization $\text{Beta}(0.5, 49.5)$.
2. **Recursive Geometric Discounting**:
   At each CI build $t$, previous evidence decays with discount factor
   $\gamma = 0.98$:
   $$\alpha_t = \gamma \, \alpha_{t-1} + y_t, \quad \beta_t = \gamma \,
   \beta_{t-1} + (1 - y_t)$$
   where $y_t = 1$ on test failure and $y_t = 0$ on test pass.
3. **Effective Memory Horizon**:
   $$N_{\text{eff}} = \sum_{k=0}^\infty \gamma^k = \frac{1}{1 - \gamma}$$
   $$N_{\text{eff}} = \frac{1}{1 - 0.98} = 50 \text{ builds}$$
4. **Bayesian Promotion Condition**:
   A test transitions from Quarantine to Blocking CI if and only if the
   posterior probability that failure rate $\theta$ is below $1.0\%$ meets or
   exceeds $95.0\%$:
   $$P(\theta \le 0.010 \mid \mathcal{D}_t) = I_{0.010}(\alpha_t, \beta_t) \ge
   0.950 \quad (95.0\%)$$

### Worked Agent Execution Transcript & Comparative Telemetry

Below is the verified execution telemetry comparing the Naive $(10, 2)$ Streak
Rule against the Bayesian Dynamic Discount Gate over a 400-build simulation of a
flaky test ($p = 0.03$):

```text
================================================================================
          CI/CD PROMOTION TELEMETRY: STREAK VS. BAYESIAN FILTER
================================================================================
[TEST SIMULATION] True Flake Rate: theta = 3.0% | Horizon: 400 Builds
[BASELINE HEURISTIC] Naive (10, 2) Streak Rule
[BAYESIAN FILTER]    Beta-Binomial (gamma = 0.98), P(theta <= 1%) >= 95%

--- RUN 001 TO 050: INITIAL QUARANTINE PHASE ---
  Build 001..012: 12 Consecutive Passes.
    * Naive Rule: Streak = 10 reached at Build 10! ==> [PROMOTED TO BLOCKING!]
    * Bayesian:   a = 0.78, b = 11.4 | P(theta <= 1%) = 28.4% ==> [QUARANTINE]
  
  Build 013: FAIL!
    * Naive Rule: Streak reset to 0; needs 2 fails to demote ==> [BLOCKING]
    * Bayesian:   a = 1.76, b = 11.2 | P(theta <= 1%) = 12.1% ==> [QUARANTINE]
  
  Build 014..035: 21 Passes, 1 Fail at Build 36.
    * Naive Rule: No 2 consecutive fails! REMAINS PROMOTED FOR 40 BUILDS!
    * Bayesian:   a = 1.62, b = 34.5 | P(theta <= 1%) = 48.7% ==> [QUARANTINE]

--- SUMMARY STATISTICS AT BUILD 400 ---
  Metric                     Naive (10, 2) Streak    Bayesian Discount Filter
  -----------------------------------------------------------------------------
  Total Time in Blocking CI  294 builds (73.5%)      6 builds (1.5%)
  False Quality Promotions   14 promotions           0 false promotions
  Pipeline Breaks Caused     9 red builds on main    0 red builds on main
  Average Time to Demote     42.6 builds             1 build (Immediate)
  -----------------------------------------------------------------------------

  ASCII DENSITY AT BUILD 400 (True theta = 3.0%):
  Bayesian Posterior Beta(alpha = 1.52, beta = 48.48):
    0.0% | **********
    1.0% | ******************** [Threshold: Only 14.2% mass to left]
    2.0% | ****************************** (Peak: ~2.5%)
    3.0% | ************************
    5.0% | ********
   10.0% | *

[DECISION VERDICT]: Bayesian filter successfully contained the 3% flaky test in
quarantine for 98.5% of its lifecycle. The naive streak rule leaked it into
blocking CI 73.5% of the time!
================================================================================
```

### Risk & Decision Evaluation
At build 400, $P(\theta \le 0.010 \mid \mathcal{D}_{400}) = 14.2\% < 95.0\%$.
The test is decisively retained in Quarantine. Under the asymmetric loss matrix
($C_{\text{leak}} = 500, C_{\text{delay}} = 10$), the expected cost of
promoting is $500 \times (1 - 0.142) = 429$, while quarantining costs
$10 \times 0.142 = 1.42$. The Bayes-optimal decision is **RETAIN_QUARANTINE**.

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it derived the dynamic
discount recursions, plotted the posterior density, and demonstrated $98.5\%$
quarantine containment.

However, as an **Epistemic Architect**, you must never deploy an autonomous
CI gating agent to production without auditing it against **four mandatory
verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Shannon entropy non-negativity / Mutual info / Promotion rule      │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Information Theoretic Invariants]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  An unconstrained AI agent frequently selects arbitrary memory discount
  parameters like $\gamma = 0.50$ (giving an effective memory horizon of only
  $N_{\text{eff}} = 2$ builds) or $\gamma = 1.00$ (infinite memory, freezing
  the gate forever). With $\gamma = 0.50$, two lucky consecutive passes reset
  the effective pseudo-counts to $(\alpha=0.25, \beta=1.75)$, tricking the
  state machine into promoting a severely broken test into blocking CI.
  Conversely, a prior with excessive pseudo-counts ($\alpha_0=100$) permanently
  traps healthy new tests in quarantine. Gate 1 guarantees that generative
  trajectories and asymptotic stationary states respect physical engineering
  realities before ingesting live CI telemetry.
* **First-Principles Mathematical Mechanism**:
  We analyze the stationary limit of the discount recursion under a test with
  constant true failure rate $p \in [0, 1]$. Since $\mathbb{E}[y_t] = p$, taking
  expectations on both sides of the update equation yields:
  $$\mathbb{E}[\alpha_\infty] = \gamma \, \mathbb{E}[\alpha_\infty] + p$$
  $$\implies \mathbb{E}[\alpha_\infty] = \frac{p}{1 - \gamma}$$
  $$\mathbb{E}[\beta_\infty] = \gamma \, \mathbb{E}[\beta_\infty] + (1 - p)$$
  $$\implies \mathbb{E}[\beta_\infty] = \frac{1 - p}{1 - \gamma}$$
  The asymptotic posterior mean is strictly unbiased:
  $$\mathbb{E}[\theta_\infty] =
  \frac{\mathbb{E}[\alpha_\infty]}{\mathbb{E}[\alpha_\infty] +
  \mathbb{E}[\beta_\infty]}$$
  $$= \frac{\frac{p}{1-\gamma}}{\frac{p}{1-\gamma} + \frac{1-p}{1-\gamma}} = p$$
  For a $3.0\%$ flaky test ($p = 0.03$) and $\gamma = 0.98$:
  $$\alpha_\infty = \frac{0.03}{0.02} = 1.50, \quad \beta_\infty =
  \frac{0.97}{0.02} = 48.50$$
  $$\implies \alpha_\infty + \beta_\infty = 50.0 = N_{\text{eff}}$$
  The prior predictive distribution for synthetic trial batches generates
  statistically stationary behavior that prevents promotion thrashing.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Asymptotic Mean Conservation: $|\mathbb{E}[\theta_\infty] - p| < 10^{-12}$
    strictly (`[PASS]`).
  * Finite Effective Memory Invariant:
    $N_{\text{eff}} = \frac{1}{1 - \gamma} == 50$ for $\gamma = 0.98$
    (`[PASS]`).
  * Total Weight Normalization:
    $\sum_{k=0}^{t-1} \gamma^k (1 - \gamma) + \gamma^t == 1.0$ (`[PASS]`).
  * Non-Degenerate Support:
    $\alpha_t > 0$ and $\beta_t > 0$ strictly (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Prior Predictive & Steady-State Sanity):
  Given discount factor gamma = 0.98 and true failure rate p = 0.03:
  1. Compute the asymptotic steady-state expectations E[alpha_inf] and
     E[beta_inf].
  2. Prove that the steady-state posterior mean equals p = 0.03 exactly.
  3. Verify effective sample size N_eff = 1 / (1 - gamma) == 50.
  4. Confirm that prior predictive draws do not generate negative rates or
     infinite variances.
  Assert exact mathematical equality. Report [PASS] or [FAIL].
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Survivorship Recovery

* **The Production Risk (Why This Gate Exists)**:
  The defining pathology of CI/CD test promotion is **survivorship bias**.
  Under discrete streak heuristics (such as the naive $(10, 2)$ rule), a broken
  test with a $3\%$ flake rate has an independent pass probability of $97\%$.
  Across hundreds of builds, the probability of encountering a lucky streak of
  10 consecutive passes approaches certainty:
  $$P(\text{Streak } \ge 10 \text{ in } 400 \text{ builds}) \approx 73.5\%$$
  The test is repeatedly promoted into blocking CI, where it randomly flakes,
  breaks developer master builds, and triggers firefighting. If Gate 2 is
  skipped, an AI agent will validate streak rules using small samples ($n=20$)
  without detecting the long-horizon survivorship failure.
* **First-Principles Mathematical Mechanism**:
  We execute $M = 5{,}000$ independent simulations of a CI pipeline running
  $T = 400$ builds for flaky tests with true failure rate
  $\theta^* \sim \text{Uniform}(0.02, 0.08)$. In each simulation $m$, we track:
  1. Promoted Fraction:
     $$\Phi_m = \frac{1}{T} \sum_{t=1}^T \mathbb{I}(\text{State}_t =
     \text{PROMOTED})$$
  2. False Promotion Rate (FPR): Fraction of simulations where an unstable test
     ($\theta^* > 0.010$) was promoted into blocking CI.
  For the Bayesian discount filter ($\gamma = 0.98, \text{Threshold} = 95\%$):
  $$\text{FPR}_{\text{Bayes}} = \frac{1}{M}\sum_{m=1}^M \mathbb{I}(\Phi_m >
  0.05) < 0.020 \quad (2.0\%)$$
  For the naive $(10, 2)$ streak heuristic:
  $$\text{FPR}_{\text{Streak}} \approx 0.735 \quad (73.5\%)$$
  The simulation proves that the Bayesian filter suppresses survivorship
  leakage by an order of magnitude.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Flake Containment Invariant: For $\theta^* = 0.03$, total blocking time
    fraction $\Phi \le 0.020$ (`[PASS]`; naive streak $\Phi \approx 0.735$
    triggers immediate `[FAIL]`).
  * Healthy Test Recovery Invariant: For a clean test ($\theta^* = 0.001$),
    time to promotion $T_{\text{promote}} \le 60$ builds with probability
    $\ge 0.950$ (`[PASS]`).
  * Rank Statistic Uniformity: Kolmogorov-Smirnov test $p$-value on posterior
    ranks across 5,000 runs satisfies $p_{\text{KS}} > 0.01$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration & Survivorship Audit):
  Run 5,000 synthetic simulations of a 400-build CI pipeline for a flaky test
  with true failure rate theta = 0.03:
  1. Compare the naive (10, 2) streak rule against the Bayesian discount filter
     (gamma = 0.98, threshold P(theta <= 0.01) >= 0.95).
  2. Measure the fraction of builds spent in blocking CI.
  3. Assert that the Bayesian filter holds false blocking time <= 2.0% while
     the naive streak rule leaks > 70% of the time.
  Report [PASS] or [FAIL].
  ```

---

### Gate 3: Mathematical Invariant Conservation (Shock & Recovery Dynamics)

* **The Production Risk (Why This Gate Exists)**:
  A test promotion filter must exhibit asymmetric memory dynamics: it must be
  **instantly demoted upon a single failure** (avoiding pipeline disruption),
  yet require **sustained, calibrated proof of stability to regain promotion**
  (preventing thrashing). Naive state machines often implement symmetric rules
  or suffer from state drift where integer counters wrap around or fail to
  register intermittent bursts. Gate 3 verifies that the analytical differential
  equations of belief collapse and recovery hold exactly.
* **First-Principles Mathematical Mechanism**:
  1. **Single-Failure Demotion Shock**:
     Suppose a healthy candidate reaches promotion threshold with
     $\alpha_0 = 0.50, \beta_0 = 49.50$, where $P(\theta \le 0.01) = 98.2\%$.
     Upon observing a single failure ($y_t = 1$):
     $$\alpha_1 = 0.98 \times 0.50 + 1 = 1.49$$
     $$\beta_1 = 0.98 \times 49.50 + 0 = 48.51$$
     We compute the regularized incomplete beta integral at $\theta_0 = 0.010$:
     $$P(\theta \le 0.010 \mid \alpha_1=1.49, \beta_1=48.51)$$
     $$= I_{0.010}(1.49, 48.51) \approx 0.7410 \quad (74.10\%)$$
     Because $74.10\% < 95.00\%$, the gate triggers an **instant demotion to
     quarantine in exactly one build**!
  2. **Analytical Recovery Requirement**:
     Under a string of consecutive passes ($y = 0$), $\alpha$ decays
     exponentially:
     $$\alpha_t = \gamma^t \, \alpha_1 = (0.98)^t \times 1.49$$
     To rebuild confidence to $P(\theta \le 0.010) \ge 0.950$, $\alpha_t$ must
     decay back to $\le 0.50$:
     $$0.98^t \times 1.49 \le 0.50 \implies t \ge \frac{\ln(0.50 /
     1.49)}{\ln(0.98)}$$
     $$t \ge \frac{-1.0919}{-0.0202} \approx 54.05 \text{ builds}$$
     The test must demonstrate $\approx 54$ consecutive passes to earn
     re-promotion.
  3. **Shannon Entropy and Information Conservation**:
     Information gain per clean run is strictly bounded by relative entropy:
     $$D_{\text{KL}}(p_t \parallel p_{t-1}) \ge 0$$
     guaranteeing monotonic uncertainty resolution during clean streaks.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Single-Failure Demotion Invariant:
    $P(\theta \le 0.01 \mid y_t=1) \le 0.750$, enforcing immediate demotion
    (`[PASS]`).
  * Recovery Threshold Invariant:
    Exact required clean passes $N_{\text{clean}} \in [53, 55]$ (`[PASS]`).
  * State Footprint Conservation:
    State is fully represented by $(\alpha_t, \beta_t, \text{state})$,
    requiring $\mathcal{O}(1)$ space and time (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Shock Collapse & Analytical Recovery Bounds):
  1. Starting from promoted state (alpha = 0.50, beta = 49.50), apply a single
     failure y = 1 under gamma = 0.98.
  2. Verify alpha jumps to 1.49 and beta to 48.51.
  3. Compute exact regularized incomplete Beta integral P(theta <= 0.01).
  4. Assert P(theta <= 0.01) <= 0.750, triggering immediate demotion.
  5. Calculate exact number of consecutive passes required to decay alpha
     back to 0.50: t = ln(0.5 / 1.49) / ln(0.98) ≈ 54 passes.
  Assert exact algebraic equality. Report [PASS] or [FAIL].
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss Optimization

* **The Production Risk (Why This Gate Exists)**:
  In continuous integration, false positive promotions and false positive
  demotions carry starkly asymmetric business costs. Leaking a flaky test into
  blocking CI halts hundreds of engineers, invalidates caches, and costs
  $C_{\text{leak}} = \$500$. Holding a stabilized test in quarantine costs only
  $C_{\text{delay}} = \$10$ (a minor delay in gating code). If an agent uses
  symmetric classification (e.g. promoting whenever posterior mean $\le 1\%$,
  or using a 50% probability threshold), it will leak tens of flaky tests into
  production. Gate 4 guarantees that promotion decisions minimize Bayes risk
  under the true asymmetric loss matrix.
* **First-Principles Mathematical Mechanism**:
  1. **Asymmetric Decision Matrix**:
     Action space $\mathcal{A} = \{\text{Promote}, \text{Quarantine}\}$.
     Parameter space $\Theta_{\text{clean}} = [0, 0.010]$ and
     $\Theta_{\text{flaky}} = (0.010, 1.000]$.
     Loss function:
     * $L(\text{Promote}, \theta \in \Theta_{\text{clean}}) = 0$
     *  $L(\text{Promote}, \theta \in \Theta_{\text{flaky}}) = C_{\text{leak}} =
       500$
     *  $L(\text{Quarantine}, \theta \in \Theta_{\text{clean}}) =
       C_{\text{delay}} = 10$
     * $L(\text{Quarantine}, \theta \in \Theta_{\text{flaky}}) = 0$
     The Bayes-optimal decision rule minimizes expected loss:
     $$\mathbb{E}[L(\text{Promote})] = C_{\text{leak}} \cdot P(\theta > 0.010
     \mid \mathcal{D})$$
     $$\mathbb{E}[L(\text{Quarantine})] = C_{\text{delay}} \cdot P(\theta \le
     0.010 \mid \mathcal{D})$$
     Promote if and only if $\mathbb{E}[L(\text{Promote})] <
     \mathbb{E}[L(\text{Quarantine})]$:
     $$500 \cdot (1 - P_{\text{clean}}) < 10 \cdot P_{\text{clean}}$$
     $$\implies P_{\text{clean}} > \frac{500}{510} \approx 98.04\%$$
     Our operational threshold of $95.0\%$ represents a cost ratio of
     $C_{\text{leak}} / C_{\text{delay}} = 19.0$.
  2. **Posterior Predictive Retrodiction**:
     We draw replicated telemetry sequences $\tilde{y}_{1:N} \sim p(\tilde{y}
     \mid \mathcal{D})$
     and verify that historical pipeline disruption rates fall within the $95\%$
     predictive interval of the retrodictive density.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Economic Risk Threshold: For $P(\theta \le 0.010) = 0.940$, action MUST be
    `QUARANTINE` (`[PASS]`; any agent recommending `PROMOTE` fails).
  * Bayes Risk Minimization:
    $\text{Action} == \arg\min_{a \in \mathcal{A}} \mathbb{E}[L(a \mid
    \mathcal{D})]$
    strictly (`[PASS]`).
  * Retrodictive Telemetry Fit: Historical failure counts fall inside central
    $95\%$ predictive envelope (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Bayesian Loss Optimization & Telemetry Retrodiction):
  Given posterior distribution Beta(alpha_t, beta_t) and asymmetric loss
  parameters C_leak = 500, C_delay = 10:
  1.  Derive the economic indifference threshold p* = C_leak / (C_leak +
     C_delay).
  2. Compute expected loss for action Promote vs action Quarantine.
  3. For a test with P(theta <= 0.01) = 0.942, assert action == QUARANTINE.
  4. For a test with P(theta <= 0.01) = 0.985, assert action == PROMOTE.
  Assert strict expected loss minimization. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Feature | Naive AI Prompting ((10, 2) Streak) | Calibrated Epistemic Specification (Bayesian Filter) |
| :--- | :--- | :--- |
| **Telemetry History** | Amnesiac: remembers only last 10 runs | Smooth recency: exponential memory ($\gamma = 0.98, N_{\text{eff}} = 50$) |
| **Survivorship Resistance** | Fails completely: $73.5\%$ false promotion rate | Robust: $<2\%$ false promotion on $3\%$ flaky tests |
| **Response to Degradation** | Sluggish: requires consecutive failures | Instant: single failure collapses posterior mass |
| **Recovery Mechanism** | Any lucky 10-pass streak resets gate | Requires $\approx 54$ clean runs to rebuild $95\%$ confidence |
| **Statistical Meaning** | Arbitrary rule of thumb | True posterior probability $P(\theta \le 0.010 \mid \mathcal{D}) \ge 0.95$ |
| **Computational Footprint** | Integer counter | Two floats $(\alpha_t, \beta_t)$; $\mathcal{O}(1)$ memory and compute |



## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An AI assistant proposes a test promotion rule:
> *"To ensure maximum quality, we require a test to pass 20 times in a row
> before promotion. Because (1 - 0.05)^20 = 35.8%, a 5% flaky test has only a
> 35.8% chance of getting promoted. This guarantees that 64.2% of broken tests
> are permanently blocked."*

**Your Task**: Explain why the AI's calculation commits the Gambler's Fallacy
and calculate the true probability of a $5\%$ flaky test getting promoted if it
runs 200 times in staging.

<details><summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **The Fallacy**: The AI calculates the probability of a 20-streak on a
   *single, isolated attempt* of 20 runs. But tests in quarantine run
   continuously across dozens of PRs. Staging tests are not tested once; they
   are tested in a rolling stream!
2. **True Probability over 200 Runs**:
   Over $N = 200$ runs with $p = 0.05$, the probability of achieving *at least
one* streak of 20 consecutive passes is approximately:
   $$P(\text{Streak } \ge 20 \text{ in } 200 \text{ runs}) \approx 1 - (1 - 0.3584)^{200 / 20} \approx 1 - (0.6416)^{10} \approx \mathbf{98.8\%}!$$
   In reality, nearly $99\%$ of broken $5\%$ flaky tests will achieve a 20-pass
streak and get promoted!
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineering team uses a rolling 30-day window to evaluate test flakiness:
`flakiness = failures_last_30_days / runs_last_30_days`.
Explain the "cliff-edge problem" of rolling windows and rewrite the
specification into an exponential decay filter with an equivalent 30-day
half-life assuming 5 builds per day.

<details><summary><b>Click here to view the solution...</b></summary>

**Analysis & Refactored Specification**:
* **The Cliff-Edge Problem**: In a strict 30-day rolling window, a failure
  that occurred on Day 1 remains at full weight for 30 days, and then abruptly
  drops off a cliff on Day 31. This causes artificial step-function jumps in
  test scores.
* **Calibrating Half-Life**:
  At 5 builds per day, a 30-day half-life equals $T_{1/2} = 150 \text{ builds}$.
  $$\gamma = e^{-\ln(2) / T_{1/2}} = e^{-0.69315 / 150} \approx \mathbf{0.99539}$$
* **Declarative Specification**:
```yaml
decay_filter:
  half_life_builds: 150
  gamma: 0.99539
  n_eff: 216.9  # 1 / (1 - gamma)
  update_step:
    on_build:
      alpha: "0.99539 * alpha + (1.0 if failed else 0.0)"
      beta: "0.99539 * beta + (1.0 if passed else 0.0)"
  metric: "alpha / (alpha + beta)"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Design an autonomous gatekeeper agent that manages three distinct tiers:
1. **Tier 1 (Quarantine)**: Non-blocking, runs on all builds.
2. **Tier 2 (Candidate)**: Non-blocking, requires $P(\theta \le 0.02) \ge 0.90$.
3. **Tier 3 (Blocking CI)**: Blocking gate, requires $P(\theta \le 0.005) \ge
   0.95$.
Specify the exact mathematical transition rules and demotion criteria.

<details><summary><b>Click here to view the solution...</b></summary>

**Multi-Tier Agent Contract**:
```yaml
three_tier_gatekeeper:
  memory_decay: 0.98
  tier_definitions:
    quarantine:
      description: "Default landing tier for new or repaired tests"
      blocking: false
      promotion_rule: "Promote to Candidate if P(theta <= 0.02 | Telemetry) >= 0.90"
      
    candidate:
      description: "Probationary tier; runs on all PRs but non-blocking"
      blocking: false
      promotion_rule: "Promote to Blocking if P(theta <= 0.005 | Telemetry) >= 0.95"
      demotion_rule: "Demote to Quarantine if P(theta <= 0.02 | Telemetry) < 0.80"
      
    blocking:
      description: "Official release gatekeeper; failure halts deployment"
      blocking: true
      demotion_rule: "Demote to Candidate immediately if P(theta <= 0.005 | Telemetry) < 0.90"
      emergency_demotion: "Demote directly to Quarantine if single build failure causes P(theta > 0.03) >= 0.50"
```
</details>



## Key Takeaways & Course Navigation

1. **Streak Heuristics Suffer from Survivorship Bias**: A $5\%$ flaky test has a
   $58.4\%$ probability of achieving a 50-pass streak across 500 builds. Never
   trust streak rules to promote tests.
2. **Structural Amnesia Ignores Context**: Streak rules treat a test with 10
   fails followed by 10 passes identically to a test with 1,000 clean runs.
3. **Bayesian Dynamic Filters Provide $O(1)$ Memory**: Exponential memory
   decay ($\gamma = 0.98$) requires storing only two numbers ($\alpha_t,
   \beta_t$) while maintaining full historical evidence weighted by recency.
4. **Immediate Demotion Protects Developers**: A single failure under the
   Bayesian filter immediately collapses posterior confidence, demoting unstable
   tests before they disrupt engineering velocity.

---

## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* ➡️ **Next**: **[Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 20**: [📖 Conceptual](../conceptual/20_test_promotion_heuristics_vs_bayes.ipynb) | [🤖 AI-Native Track](README.md) | [🐍 Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb) | [🔵 R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)

